# 01 · Inspección y calidad de la señal

**Proyecto final · Reactividad alfa en EEG durante la alternancia de ojos abiertos y cerrados**

| | |
|---|---|
| Dueño | A · Moisés |
| Etapas de la rúbrica | Etapa 1 + selección de canales (15 %) |
| Entregable técnico | `cargar_raw()`, `tabla_bloques()`, `marcar_artefactos()`, `resumen_artefactos()` |
| Figuras | fig01_senal_original (canales elegidos + VEOG con bloques EO/EC sombreados) |

Reglas: este notebook solo lo edita su dueño. Se abre desde GitHub, se guarda con *Archivo → Guardar una copia en GitHub* en tu rama
(nunca en `main`) y antes de guardar se borran las salidas (*Editar → Borrar todas las salidas*). Las figuras se guardan con
`alfa.guardar_figura(fig, "figNN_nombre")` en la carpeta `figuras/` de Drive.

Memoria: el registro completo a 2500 Hz ocupa ~1.2 GB. Aquí se carga con `preload=False` y solo se leen a memoria
los canales que se usan en cada paso.

## 0. Preparación

In [ ]:
# ── Preparación (igual en todos los notebooks; no editar) ──────────────
RAMA = "main"   # cámbiala solo si necesitas un alfa.py que aún está en un PR

import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/repo"
    URL = "https://github.com/moiseiseiseis/Alpha-reactivity-during-resting-state-EEG.git"
    if not os.path.isdir(REPO):
        !git clone -q -b {RAMA} {URL} {REPO}
    else:
        !git -C {REPO} fetch -q origin {RAMA} && git -C {REPO} checkout -q {RAMA} && git -C {REPO} reset -q --hard origin/{RAMA}
    %pip install -q mne
    sys.path.insert(0, f"{REPO}/src")
else:
    sys.path.insert(0, os.path.abspath("../src"))

import importlib
import alfa
importlib.reload(alfa)   # recoge cambios de alfa.py sin reiniciar el entorno

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

R = alfa.rutas()
print("Datos en:", R["datos"])
print("Registro encontrado:", R["vhdr"].exists())

In [ ]:
# Parámetros en uso: todo sale de CONFIG (no escribir números a mano en el notebook)
cfg = alfa.CONFIG
alfa.imprimir_config(cfg)
mne.set_log_level("ERROR")
COLOR = {"EC": "tab:blue", "EO": "tab:orange"}

## 1. Características del registro

Se carga con `alfa.cargar_raw()`: VEOG queda como canal `eog`, se verifica que el encabezado declare µV en todos los
canales y se agregan las posiciones 10-05. La referencia, la tierra y los filtros salen del `eeg.json` y del
`channels.tsv` de BIDS, que MNE no lee.

In [ ]:
raw = alfa.cargar_raw(R["vhdr"], preload=False)

meta = pd.read_json(R["datos"] / "sub-010002_task-resting_eeg.json", typ="series")
canales_tsv = pd.read_csv(R["datos"] / "sub-010002_task-resting_channels.tsv", sep="\t")

tipos = pd.Series(raw.get_channel_types()).value_counts().to_dict()
caracteristicas = pd.Series({
    "Frecuencia de muestreo (Hz)": raw.info["sfreq"],
    "Duración (s)": round(raw.times[-1], 1),
    "Muestras por canal": raw.n_times,
    "Canales por tipo": tipos,
    "Unidades en el archivo": sorted(set(raw._orig_units.values())),
    "Referencia": meta["EEGReference"],
    "Tierra": meta["EEGGround"],
    "Sistema de colocación": meta["EEGPlacementScheme"],
    "Equipo": f'{meta["Manufacturer"]} {meta["ManufacturersModelName"]}',
    "Línea eléctrica (Hz)": raw.info["line_freq"],
    "Filtros de software": meta["SoftwareFilters"],
    "Filtros de hardware (channels.tsv)": canales_tsv[["low_cutoff", "high_cutoff"]].drop_duplicates().to_dict("records")
        if {"low_cutoff", "high_cutoff"} <= set(canales_tsv.columns) else "ver channels.tsv",
    "Estado de los canales (channels.tsv)": canales_tsv["status"].value_counts().to_dict()
        if "status" in canales_tsv.columns else "n/a",
})
caracteristicas.to_frame("valor")

**Lectura.** 61 canales EEG y un VEOG a 2500 Hz, unos 17 min. La referencia es FCz, que no aparece como canal: todos los
canales están medidos contra ese electrodo. AFz es la tierra del amplificador, pero sí se registra como canal (Ch22); su
señal se revisa en la §3. MNE devuelve los datos en voltios, así que en todo el proyecto se multiplica por 1e6 para µV.

## 2. Marcadores y bloques

`tabla_bloques()` define un bloque como una racha de marcadores de condición iguales y consecutivos. `fin_s` es el último
marcador más la separación típica entre marcadores (2 s).

In [ ]:
bloques = alfa.tabla_bloques(raw)
huecos = bloques["inicio_s"].iloc[1:].to_numpy() - bloques["fin_s"].iloc[:-1].to_numpy()
print(f"{len(bloques)} bloques; duración {bloques.duracion_s.min():.1f}-{bloques.duracion_s.max():.1f} s; "
      f"huecos entre bloques {huecos.min():.2f}-{huecos.max():.2f} s; "
      f"S 2 antes de cada bloque: {(1000 * (bloques.inicio_s - bloques.cambio_s)).min():.0f}-"
      f"{(1000 * (bloques.inicio_s - bloques.cambio_s)).max():.0f} ms")
print(f"Sin condición: 0-{bloques.inicio_s.iloc[0]:.1f} s al inicio y {bloques.fin_s.iloc[-1]:.1f}-{raw.times[-1]:.1f} s al final")
bloques.round(3)

**Lo que dicen los marcadores**

| `.vmrk` (lo que lee MNE) | `events.tsv` (BIDS) | Significado | Veces |
|---|---|---|---|
| `S  1` | `Comment/no USB Connection` | inicio del archivo | 1 |
| `S  2` | `Stimulus/S  1` | cambio de bloque, ~10 ms antes del primer marcador | 17 |
| `S  3` | `Stimulus/S200` | ojos abiertos (EO), cada 2 s | 240 |
| `S  4` | `Stimulus/S210` | ojos cerrados (EC), cada 2 s | 240 |

Los marcadores de condición se repiten cada 2 s dentro del bloque, así que no marcan inicios. Hay 16 bloques de 60 s con 30
marcadores cada uno, alternando y empezando con EC a los 6.4 s. Cada `S 2` llega 9-10 ms antes del primer marcador de su
bloque, lo que confirma que marca el cambio de condición; hay 17 y no 16 porque el primer bloque tiene dos (3.99 s y 6.38 s).
Entre bloques quedan 1.6-2.6 s sin condición y, después del último, unos 25 s. Esos tramos no entran al análisis.
El `.vmrk` cuenta las muestras desde 1 y el `events.tsv` desde 0.

## 3. Inspección de la señal en el tiempo

Para recorrer los 17 min sin cargar 1.2 GB se arma una copia ligera: todos los canales pasan por `alfa.acondicionar()`
(1-40 Hz, 250 Hz) en grupos de 16. Esta copia solo se usa para inspeccionar; la Figura 1 usa la señal original.

In [ ]:
def acondicionar_por_grupos(raw, canales, cfg, tam=16):
    """acondicionar() por grupos de canales para no tener el registro completo en memoria."""
    partes = [alfa.acondicionar(raw.copy().pick(canales[i:i + tam]), cfg) for i in range(0, len(canales), tam)]
    out = partes[0]
    if len(partes) > 1:
        out.add_channels(partes[1:])
    return out

EEG = [c for c, t in zip(raw.ch_names, raw.get_channel_types()) if t == "eeg"]
ligero = acondicionar_por_grupos(raw, EEG + ["VEOG"], cfg)
ligero

### 3.1 Amplitud pico a pico por ventana

Cada celda es el pico a pico de un canal en una ventana de 2 s. Las franjas de arriba marcan los bloques EC y EO.

In [ ]:
sf = ligero.info["sfreq"]
w = int(2 * sf)
x = ligero.get_data() * 1e6
nw = x.shape[1] // w
pp = np.ptp(x[:, : nw * w].reshape(len(ligero.ch_names), nw, w), axis=2)
t_pp = np.arange(nw) * 2 + 1

fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(np.log10(pp), aspect="auto", cmap="magma", extent=[0, nw * 2, len(ligero.ch_names) - 0.5, -0.5])
ax.set_yticks(range(len(ligero.ch_names)))
ax.set_yticklabels(ligero.ch_names, fontsize=6)
for b in bloques.itertuples():
    ax.axvspan(b.inicio_s, b.fin_s, ymin=1.0, ymax=1.02, color=COLOR[b.condicion], clip_on=False)
ax.set_xlabel("Tiempo (s)")
ax.set_title("Amplitud pico a pico por ventana de 2 s (1-40 Hz)")
fig.colorbar(im, ax=ax, label="log10 pico a pico (µV)")
plt.show()

malas = (pd.DataFrame(pp.T, columns=ligero.ch_names).assign(t_s=t_pp)
         .melt(id_vars="t_s", var_name="canal", value_name="pp_uv")
         .query("canal in @cfg['canales']").nlargest(10, "pp_uv"))
malas.round(1)

In [ ]:
# Resumen por canal: dispersión, ventanas que pasan el umbral, ruido muscular (20-40 Hz) y 50 Hz.
from scipy.signal import welch
f, pxx = welch(ligero.get_data(picks=EEG) * 1e6, fs=sf, nperseg=int(2 * sf))
banda = lambda a, b: pxx[:, (f >= a) & (f <= b)].sum(axis=1) * (f[1] - f[0])

umbral = cfg["artefactos"]["umbral_pp_uv"]
calidad = pd.DataFrame({
    "de_uv": x[: len(EEG)].std(axis=1),
    "pp_p99_uv": np.percentile(pp[: len(EEG)], 99, axis=1),
    f"ventanas_pp>{umbral:.0f}": (pp[: len(EEG)] > umbral).sum(axis=1),
    "potencia_20_40_uv2": banda(20, 40),
}, index=EEG).round(1)
calidad.loc[cfg["canales"]]

In [ ]:
# 50 Hz en la señal original (sin filtrar), 60 s de un bloque EO en los canales de análisis.
b2 = bloques.iloc[1]
seg = raw.copy().pick(cfg["canales"]).crop(b2.inicio_s, b2.inicio_s + 60).load_data()
psd = seg.compute_psd(fmin=1, fmax=100, n_fft=int(4 * seg.info["sfreq"]))
fr, p = psd.freqs, psd.get_data().mean(axis=0) * 1e12
vecinos = p[((fr >= 45) & (fr <= 48)) | ((fr >= 52) & (fr <= 55))].mean()
print(f"PSD en 50 Hz / vecinos (45-48 y 52-55 Hz): {p[np.argmin(abs(fr - 50))] / vecinos:.1f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(fr, p, color="k")
ax.axvline(50, ls="--", color="gray")
ax.set_xlabel("Frecuencia (Hz)"); ax.set_ylabel("PSD (µV²/Hz)")
ax.set_title(f"Señal original, promedio de canales de análisis, bloque {b2.bloque} ({b2.condicion})")
plt.show()

In [ ]:
def ver(inicio_s, duracion=20, canales=None, datos=None, escala=100):
    """Trazas apiladas de un tramo, con los bloques sombreados. escala = µV entre trazas."""
    datos = ligero if datos is None else datos
    canales = (cfg["canales"] + ["Fp1", "AFz", "VEOG"]) if canales is None else canales
    tramo = datos.copy().pick(canales).crop(inicio_s, min(inicio_s + duracion, datos.times[-1]))
    y = tramo.get_data() * 1e6
    y = y - y.mean(axis=1, keepdims=True)
    t = tramo.times + inicio_s
    fig, ax = plt.subplots(figsize=(13, 0.45 * len(canales) + 1.5))
    for i, c in enumerate(canales):
        ax.plot(t, y[i] / (4 if c == "VEOG" else 1) - i * escala, lw=0.6, color="k")
    ax.set_yticks([-i * escala for i in range(len(canales))])
    ax.set_yticklabels([c + (" (÷4)" if c == "VEOG" else "") for c in canales])
    for b in bloques.itertuples():
        ax.axvspan(b.inicio_s, b.fin_s, color=COLOR[b.condicion], alpha=0.12, lw=0)
    for a in datos.annotations:
        if a["description"].startswith("BAD") and inicio_s <= a["onset"] <= inicio_s + duracion:
            ax.axvspan(a["onset"], a["onset"] + a["duration"], color="red", alpha=0.25, lw=0)
    ax.set_xlim(t[0], t[-1]); ax.set_xlabel("Tiempo (s)")
    ax.set_title(f"{inicio_s:.0f}-{inicio_s + duracion:.0f} s · separación {escala} µV · azul EC, naranja EO, rojo BAD_")
    plt.show()

ver(56, 24)      # transición EC → EO (bloque 1 → 2)
ver(1000, 22)    # final del registro, después del último bloque

### 3.2 Lo que se observa

- **Canales posteriores:** limpios. En los ocho canales de análisis, el pico a pico de 2 s tiene un percentil 99 de
  99-124 µV y, dentro de los bloques, ninguna ventana supera 150 µV.
- **Deriva lenta en cada transición:** después de cada cambio de condición, VEOG y los canales posteriores se desplazan
  varias decenas de µV y tardan unos 20 s en volver a la línea base (panel superior de la Figura 1). El filtro de
  hardware empieza en 0.016 Hz, así que esa deriva queda en el registro; el pasa-altas de 1 Hz de B la quita.
- **O1:** un escalón brusco hacia los 83.5 s (bloque 2, EO), típico de un cambio de impedancia del electrodo.
- **Final del registro (~1010-1012 s):** un transitorio grande y simultáneo en todos los canales (hasta 400 µV en Oz,
  716 µV en AFz, 1200 µV en VEOG). Es el pico de 436 µV en Oz que aparecía en la exploración inicial. Ocurre después del último
  bloque (997.3 s), así que no afecta el análisis; probablemente es movimiento al terminar la tarea.
- **Parpadeos:** grandes en VEOG, Fp1 y Fp2 y casi ausentes con los ojos cerrados. También aparecen, atenuados, en los
  canales posteriores (§5), probablemente a través de la referencia FCz.
- **Transiciones:** al abrir o cerrar los ojos hay deflexiones grandes en VEOG, Fp1 y AFz que llegan invertidas y
  atenuadas a los canales posteriores (por ejemplo, entre 66 y 71 s en `ver(56, 24)`). Junto con la deriva lenta, esto
  respalda descartar `recorte_transicion_s` = 5 s al inicio de cada bloque.
- **50 Hz:** el pico de línea es pequeño frente a sus vecinos y, de todos modos, queda fuera de la banda que deja pasar el
  filtro de B (1-40 Hz).
- **Ruido muscular (20-40 Hz):** es mayor en temporales (T7, TP7, FC5) y en los canales inferiores (PO9, PO10, P7) que en la
  línea media posterior.
- **AFz:** a pesar de ser la tierra, su señal se parece a la de los canales frontales vecinos; no se usa en el análisis.

**Respuestas de la Etapa 1**

- *¿La señal es similar en todo el registro?* En los canales posteriores sí: la amplitud se mantiene en el mismo rango
  durante los 16 bloques, con la diferencia esperada entre EC y EO. Lo único fuera de rango es el transitorio final, que
  queda fuera de los bloques.
- *¿Hay segmentos que comprometan el análisis?* Pocos: los parpadeos de EO y algunos segundos con amplitud alta (§5).
- *¿Qué artefactos hay?* Oculares (parpadeos y movimientos al cambiar de condición), musculares en canales temporales e
  inferiores, y un transitorio de movimiento al final.
- *¿Por qué importa elegir bien canales y segmentos?* Porque el alfa con los ojos cerrados es posterior y los artefactos
  oculares y musculares tienen potencia en frecuencias vecinas; un canal frontal o temporal mediría más artefacto que alfa.

## 4. Selección y justificación de canales

El criterio principal es fisiológico: el alfa en reposo con los ojos cerrados es máximo en regiones occipitales y
parieto-occipitales, sobre la corteza visual (fundamento de D, §4 del documento maestro). Con la señal se comprueba esa
distribución y la calidad de cada canal. Para no elegir canales *por* el efecto que se quiere medir (selección circular), la
razón EC/EO de abajo solo se usa para confirmar la región, no para escoger los canales uno por uno.

In [ ]:
cfg_todos = alfa.variante(canales=EEG, excluir_artefactos=False)
pot = alfa.potencia_alfa(ligero, bloques, cfg_todos)
pot = pot[pot["dentro"]]
por_bloque = pot.groupby(["canal", "bloque", "condicion"])["alfa_abs"].mean().reset_index()
por_bloque["par"] = (por_bloque["bloque"] + 1) // 2
pares = por_bloque.pivot_table(index=["canal", "par"], columns="condicion", values="alfa_abs")

por_canal = pot.groupby(["canal", "condicion"])["alfa_abs"].mean().unstack()
por_canal["razon_ec_eo"] = por_canal["EC"] / por_canal["EO"]
por_canal["pares_ec>eo"] = (pares["EC"] > pares["EO"]).groupby("canal").sum()
por_canal["razon_min_par"] = (pares["EC"] / pares["EO"]).groupby("canal").min()
por_canal["potencia_20_40_uv2"] = calidad["potencia_20_40_uv2"]
por_canal["elegido"] = por_canal.index.isin(cfg["canales"])
por_canal.sort_values("razon_ec_eo", ascending=False).round(2).head(25)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
info = ligero.copy().pick(EEG).info
orden = [por_canal.loc[c] for c in EEG]
mascara = np.array([c in cfg["canales"] for c in EEG])
for ax, (col, titulo, cmap) in zip(axes, [
        ("EC", "Alfa EC (µV², log10)", "viridis"),
        ("razon_ec_eo", "Razón EC/EO (log10)", "RdBu_r"),
        ("potencia_20_40_uv2", "Potencia 20-40 Hz (µV², log10)", "magma")]):
    v = np.log10([o[col] for o in orden])
    im, _ = mne.viz.plot_topomap(v, info, axes=ax, show=False, cmap=cmap, mask=mascara,
                                 mask_params=dict(markersize=6, markerfacecolor="w"))
    ax.set_title(titulo)
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle("Distribución del alfa y del ruido muscular (blanco: canales elegidos)")
plt.show()

### Canales elegidos

`CONFIG["canales"]` = **O1, Oz, O2, PO7, PO3, POz, PO4, PO8** (occipitales y parieto-occipitales, simétricos respecto a la
línea media).

- **Por qué estos.** Cubren la región donde se espera el alfa de reposo con los ojos cerrados. En este registro, la razón
  EC/EO de esos ocho canales va de 3.6 a 5.3 y es mayor que 1 en los 8 pares de bloques en cada canal; la topografía muestra
  el máximo en esa misma zona y una caída hacia adelante, hasta 0.9-1.0 en Fp1 y Fp2.
- **Por qué no PO9 y PO10.** Están en posición inferior y lateral, sobre la musculatura del cuello. PO10 tiene la potencia de
  20-40 Hz más alta de los canales posteriores y la razón EC/EO más baja en su peor par (1.36). Se quitan los dos para mantener
  la simetría.
- **Por qué no la fila P (P3, Pz, P4).** Están más lejos del máximo y su reactividad es menor (razón 2.3-2.9).
- **Qué se pierde.** La información de regiones parietales y temporales, y con ella la posibilidad de comparar la
  topografía completa. Como el análisis promedia los ocho canales, una diferencia entre hemisferios tampoco se verá.
- **Para la Etapa 5.** La lista provisional (P3, Pz, P4, PO9, O1, Oz, O2, PO10) sirve como variante de canales en el
  notebook 04: `alfa.variante(canales=[...])`.

Los frontales cercanos a FCz (FC1, FC2, Cz, Fz) tienen amplitudes muy bajas porque están junto a la referencia; eso
también los descarta para medir alfa con este montaje.

## 5. Artefactos y segmentos excluidos

`alfa.marcar_artefactos(raw, cfg)` agrega anotaciones `BAD_` (parámetros en `cfg["artefactos"]`):

| Anotación | Criterio |
|---|---|
| `BAD_amplitud` | en segmentos de 1 s, algún canal de análisis supera `umbral_pp_uv` de pico a pico (señal filtrada 1-40 Hz) |
| `BAD_plano` | algún canal de análisis por debajo de `plano_pp_uv` (desconexión) |
| `BAD_parpadeo` | ±0.4 s alrededor de cada evento de `find_eog_events` en VEOG |
| `BAD_manual` | filas de `resultados/segmentos_malos_manual.csv` (`inicio_s, fin_s, canales, motivo`), si existe |

`potencia_alfa()` descarta las ventanas que tocan una anotación `BAD_` cuando `excluir_artefactos=True`.

In [ ]:
raw = alfa.marcar_artefactos(raw, cfg)
ligero.set_annotations(raw.annotations)
tabla_artefactos = alfa.resumen_artefactos(raw, bloques)
tabla_artefactos.to_csv(R["resultados"] / "artefactos_resumen.csv", index=False)
tabla_artefactos

In [ ]:
# ¿Por qué find_eog_events encuentra pocos eventos? Se cuentan por condición.
ev = mne.preprocessing.find_eog_events(ligero, ch_name="VEOG")
t_ev = ev[:, 0] / sf
cuenta = {c: sum(((t_ev >= b.inicio_s) & (t_ev < b.fin_s)).sum() for b in bloques[bloques.condicion == c].itertuples())
          for c in ["EO", "EC"]}
cuenta["fuera de bloques"] = len(t_ev) - sum(cuenta.values())
print("Eventos oculares detectados:", len(t_ev), cuenta)

# ¿Llegan los parpadeos a los canales posteriores? Promedio alineado al pico del VEOG.
ep = mne.Epochs(ligero, ev, tmin=-0.5, tmax=0.5, baseline=(-0.5, -0.3), picks="all",
                reject_by_annotation=False, preload=True)
prom = ep.average(picks="all")
amp = pd.Series(np.ptp(prom.get_data(picks=cfg["canales"] + ["Fp1", "VEOG"]) * 1e6, axis=1),
                index=cfg["canales"] + ["Fp1", "VEOG"]).round(1)
print("Pico a pico del parpadeo promedio (µV):"); print(amp.to_string())

In [ ]:
# Cuántas ventanas de potencia_alfa quedan por bloque con y sin exclusión
ac = ligero.copy().pick(cfg["canales"])
n_con = alfa.resumen_por_bloque(alfa.potencia_alfa(ac, bloques, cfg))[["bloque", "condicion", "n_ventanas"]]
n_sin = alfa.resumen_por_bloque(alfa.potencia_alfa(ac, bloques, alfa.variante(excluir_artefactos=False)))
n_con["sin_exclusion"] = n_sin["n_ventanas"].to_numpy()
n_con["excluidas_pct"] = (100 * (1 - n_con["n_ventanas"] / n_con["sin_exclusion"])).round(1)
n_con

### Segmentos excluidos

- **Umbral de amplitud de 100 µV.** Se eligió con la distribución de pico a pico de la §3: con 80 µV se marcarían unos
  37 s de EC, que en su mayoría son alfa de gran amplitud (el fenómeno que se quiere medir). Con 100 µV se marca 1 s de EC
  y 10 s de EO, y lo que se marca son deflexiones lentas y parpadeos, no ráfagas de alfa.
- **Parpadeos.** `find_eog_events` detecta 61 eventos: 34 en EO, ninguno en EC y el resto en las transiciones. No es que
  el umbral automático falle: con los ojos cerrados no hay parpadeos y, con la vista fija en el LED, el participante parpadea
  poco. El parpadeo promedio mide ~430 µV en VEOG y ~50-58 µV en los canales posteriores. Casi toda su energía está por
  debajo de 4 Hz, así que contamina poco la banda alfa, pero se excluye por precaución.
- **Cuánto se pierde.** Alrededor del 0.4 % del tiempo en EC y del 6 % en EO (tabla de arriba). Por la ventana de 2 s con 50 %
  de solapamiento, en EO se pierde una fracción mayor de ventanas que de tiempo: entre 9 y 23 % por bloque.
- **Sesgo posible.** La exclusión afecta casi solo a EO. Si los parpadeos aumentaran la potencia alfa medida en EO, quitarlos
  haría crecer la razón EC/EO. Por eso D compara con y sin exclusión en la Etapa 5.
- **Revisión manual.** Si en el recorrido con `ver()` aparece algo que el umbral no detecta, se agrega una fila a
  `resultados/segmentos_malos_manual.csv` y se vuelve a correr `marcar_artefactos()`.

## 6. Figura 1 · Señal original con condiciones

Señal **sin filtrar** (solo se resta la media de cada canal en el tramo). Arriba, el registro completo en Oz y VEOG con los
16 bloques; abajo, 40 s alrededor de la primera transición EC → EO en los canales elegidos.

In [ ]:
orig = raw.copy().pick(cfg["canales"] + ["VEOG"]).load_data()
t0, t1 = bloques.fin_s.iloc[0] - 22, bloques.inicio_s.iloc[1] + 18
fig = plt.figure(figsize=(13, 10))
gs = fig.add_gridspec(3, 1, height_ratios=[1, 1, 3.2], hspace=0.45)

dec = 25  # solo para dibujar 17 min sin 2.5 M puntos (no se usa para calcular nada)
for k, ch in enumerate(["Oz", "VEOG"]):
    ax = fig.add_subplot(gs[k])
    y = orig.get_data(picks=ch)[0] * 1e6
    y = y - np.median(y)
    ax.plot(orig.times[::dec], y[::dec], lw=0.3, color="k")
    for b in bloques.itertuples():
        ax.axvspan(b.inicio_s, b.fin_s, color=COLOR[b.condicion], alpha=0.18, lw=0)
    ax.axvspan(t0, t1, color="none", ec="red", lw=1.2)
    ax.set_xlim(0, orig.times[-1]); ax.set_ylabel(f"{ch} (µV)")
    lim = np.percentile(abs(y), 99.9) * 1.2
    ax.set_ylim(-lim, lim)
    if k == 0:
        ax.set_title("Registro completo: bloques EC (azul) y EO (naranja); el recuadro rojo es el tramo de abajo")
ax.set_xlabel("Tiempo (s)")

ax = fig.add_subplot(gs[2])
tramo = orig.copy().crop(t0, t1)
y = tramo.get_data() * 1e6
y = y - y.mean(axis=1, keepdims=True)
esc = 80
for i, ch in enumerate(tramo.ch_names):
    div = 5 if ch == "VEOG" else 1
    ax.plot(tramo.times + t0, y[i] / div - i * esc, lw=0.5, color="k")
ax.set_yticks([-i * esc for i in range(len(tramo.ch_names))])
ax.set_yticklabels([c + (" (÷5)" if c == "VEOG" else "") for c in tramo.ch_names])
for b in bloques.itertuples():
    ax.axvspan(b.inicio_s, b.fin_s, color=COLOR[b.condicion], alpha=0.15, lw=0)
    if t0 < b.inicio_s < t1:
        ax.axvline(b.inicio_s, color="gray", ls="--", lw=0.8)
ax.set_xlim(t0, t1); ax.set_xlabel("Tiempo (s)")
ax.set_ylabel(f"Canal (separación {esc} µV)")
ax.set_title("Transición EC → EO (bloque 1 → 2), señal original sin filtrar")
ax.plot([t1 - 1.5, t1 - 1.5], [esc * 0.5, esc * 1.5 - 0.0], color="k", lw=2, clip_on=False)
ax.text(t1 - 1.3, esc, f"{esc} µV", va="center", fontsize=8)
alfa.guardar_figura(fig, "fig01_senal_original")
plt.show()

**Figura 1.** Señal EEG original (sin filtrar, µV) del registro `sub-010002`. Arriba: Oz y VEOG durante todo el registro,
con los bloques de ojos cerrados (EC, azul) y abiertos (EO, naranja) según los marcadores. Abajo: los ocho canales de análisis
y el VEOG (dividido entre 5) en los 40 s alrededor de la primera transición EC → EO. En los bloques EO aparecen parpadeos
en VEOG; en EC no. Cada cambio de condición produce una deflexión grande seguida de una deriva lenta de la línea base, y en
el tramo de abajo la actividad rápida de los canales posteriores tiene más amplitud en EC que en EO.

## Interpretación breve

En lo observado, el registro es estable en los canales posteriores durante los 16 bloques: la amplitud se mantiene en el
mismo rango, sin saturación ni canales planos, y la potencia de 8-13 Hz es mayor en EC que en EO en todos los canales
posteriores y en los 8 pares de bloques (§4). Los artefactos son sobre todo oculares (parpadeos en EO, deflexiones y deriva
lenta en cada transición), un escalón aislado en O1 y un transitorio grande después del último bloque. Con el criterio de exclusión se pierde menos del 1 % del tiempo en EC y cerca del 6 % en EO. La
interpretación fisiológica de estas diferencias corresponde a las secciones de C y D.